In [ ]:
# Databricks notebook source
# Notebook: check_bcb_consecutive_failures
# Purpose: Runs only when ingest_bcb_exchange_rates fails; raises on the 3rd consecutive failed day
#          so that the task's native on_failure email is sent.

from pyspark.sql import SparkSession
from grupo10.utils.logger import get_logger
from grupo10.utils.alerting import get_consecutive_failure_streak

spark = SparkSession.builder.getOrCreate()

# 1. Catalog and schema parameters
dbutils.widgets.text("catalog", "lh_nautical")
dbutils.widgets.text("schema", "prod")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

# 2. Runs only when 'ingest_bcb_exchange_rates' has failed (run_if: AT_LEAST_ONE_FAILED).
# Fails on purpose only on the 3rd consecutive failure, to trigger this task's native
# on_failure email — no email is sent directly from here.
logger = get_logger("check_bcb_consecutive_failures")
TASK_NAME = "ingest_bcb_exchange_rates"
RUN_HISTORY_TABLE = f"{catalog}.{schema}._alert_run_history"
CONSECUTIVE_FAILURE_THRESHOLD = 3

streak = get_consecutive_failure_streak(spark, RUN_HISTORY_TABLE, TASK_NAME)
logger.info(f"Consecutive failure streak for '{TASK_NAME}': {streak}")

if streak >= CONSECUTIVE_FAILURE_THRESHOLD:
    raise Exception(
        f"'{TASK_NAME}' has failed for {streak} consecutive scheduled days — escalating alert."
    )

logger.info(f"Streak below threshold ({CONSECUTIVE_FAILURE_THRESHOLD}), no alert needed.")